In [1]:
import importlib.util
import json
import warnings
from pathlib import Path
from typing import Dict, Optional, Tuple

import torch
import torch.nn.functional as F
from safetensors.torch import save_file
from torchvision.io import read_video


DATASET_ROOT = Path("/home/azureuser/datasets/navier_fast")
WAN_REPO_ROOT = Path("/home/azureuser/physics/navier/Wan2.2")
WAN_CHECKPOINT_DIR = Path("/home/azureuser/Wan2.2-TI2V-5B")
WAN_TI2V_5B_VAE_CHECKPOINT = "Wan2.2_VAE.pth"
WAN_TI2V_5B_VAE_STRIDE = (4, 16, 16)
DEFAULT_VIDEO_HEIGHT = 240
DEFAULT_VIDEO_WIDTH = 480
DEFAULT_WAN_EXPECTED_SIZE = (DEFAULT_VIDEO_HEIGHT, DEFAULT_VIDEO_WIDTH)


def _load_wan2_2_vae_class(wan_repo_root: str | Path = WAN_REPO_ROOT):
    """Load Wan2.2's VAE class without importing wan/__init__.py extras."""
    module_path = Path(wan_repo_root) / "wan" / "modules" / "vae2_2.py"
    if not module_path.exists():
        raise FileNotFoundError(f"Wan2.2 VAE module not found: {module_path}")

    spec = importlib.util.spec_from_file_location("wan_vae2_2_local", module_path)
    if spec is None or spec.loader is None:
        raise ImportError(f"Could not create import spec for {module_path}")
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module.Wan2_2_VAE


def load_wan_ti2v_5b_vae(
    checkpoint_dir: str | Path = WAN_CHECKPOINT_DIR,
    wan_repo_root: str | Path = WAN_REPO_ROOT,
    device: Optional[str | torch.device] = None,
    dtype: torch.dtype = torch.float32,
):
    """Load the Wan2.2 TI2V-5B VAE from /home/azureuser/Wan2.2-TI2V-5B."""
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu") if device is None else torch.device(device)
    checkpoint_path = Path(checkpoint_dir) / WAN_TI2V_5B_VAE_CHECKPOINT
    if not checkpoint_path.exists():
        raise FileNotFoundError(f"VAE checkpoint not found: {checkpoint_path}")

    Wan2_2_VAE = _load_wan2_2_vae_class(wan_repo_root)
    vae = Wan2_2_VAE(
        vae_pth=str(checkpoint_path),
        dtype=dtype,
        device=device,
    )
    return vae


def load_video_for_wan_vae(
    video_path: str | Path,
    device: Optional[str | torch.device] = None,
    expected_size: Tuple[int, int] = DEFAULT_WAN_EXPECTED_SIZE,
    resize: bool = False,
) -> Tuple[torch.Tensor, Dict]:
    """Read an mp4 and return Wan-normalized video shaped (C, T, H, W)."""
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu") if device is None else torch.device(device)
    video_path = Path(video_path)
    if not video_path.exists():
        raise FileNotFoundError(video_path)

    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", category=UserWarning, module="torchvision")
        frames, _, info = read_video(str(video_path), pts_unit="sec", output_format="TCHW")
    if frames.numel() == 0:
        raise ValueError(f"No video frames decoded from {video_path}")

    if frames.shape[1] < 3:
        frames = frames.repeat(1, 3, 1, 1)
    frames = frames[:, :3]
    frames = frames.float().div_(255.0) if frames.dtype == torch.uint8 else frames.float().clamp_(0.0, 1.0)

    target_h, target_w = expected_size
    if tuple(frames.shape[-2:]) != (target_h, target_w):
        if not resize:
            raise ValueError(
                f"Expected {expected_size} video frames, got {tuple(frames.shape[-2:])} for {video_path}"
            )
        frames = F.interpolate(frames, size=expected_size, mode="bicubic", align_corners=False).clamp_(0.0, 1.0)

    # Wan preprocessing uses TF.to_tensor(...).sub_(0.5).div_(0.5).
    video = frames.mul_(2.0).sub_(1.0).permute(1, 0, 2, 3).contiguous().to(device)
    return video, dict(info)


@torch.no_grad()
def encode_video_to_wan_latents(
    video_path: str | Path,
    vae=None,
    checkpoint_dir: str | Path = WAN_CHECKPOINT_DIR,
    wan_repo_root: str | Path = WAN_REPO_ROOT,
    device: Optional[str | torch.device] = None,
    vae_dtype: torch.dtype = torch.float32,
    save_dtype: torch.dtype = torch.float16,
    output_path: Optional[str | Path] = None,
    overwrite: bool = False,
    expected_size: Tuple[int, int] = DEFAULT_WAN_EXPECTED_SIZE,
    resize: bool = False,
) -> Dict[str, str | int | float | Tuple[int, ...]]:
    """Encode one dataset mp4 and save latents.safetensors beside it."""
    video_path = Path(video_path)
    output_path = video_path.with_name("latents.safetensors") if output_path is None else Path(output_path)
    if output_path.exists() and not overwrite:
        return {"status": "skipped", "reason": "exists", "video": str(video_path), "latents": str(output_path)}

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu") if device is None else torch.device(device)
    if vae is None:
        vae = load_wan_ti2v_5b_vae(checkpoint_dir, wan_repo_root, device=device, dtype=vae_dtype)

    video, video_info = load_video_for_wan_vae(video_path, device=device, expected_size=expected_size, resize=resize)
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", category=FutureWarning, message=".*torch.cuda.amp.autocast.*")
        latents = vae.encode([video])[0].detach().to("cpu", dtype=save_dtype).contiguous()

    metadata = {
        "source_video": video_path.name,
        "vae": "Wan2.2 TI2V-5B VAE",
        "vae_checkpoint": str(Path(checkpoint_dir) / WAN_TI2V_5B_VAE_CHECKPOINT),
        "vae_stride": json.dumps(WAN_TI2V_5B_VAE_STRIDE),
        "normalization": "uint8 [0,255] -> [0,1] -> [-1,1]",
        "input_shape_cthw": json.dumps(tuple(video.shape)),
        "latent_shape_cthw": json.dumps(tuple(latents.shape)),
        "latent_dtype": str(latents.dtype),
        "video_info": json.dumps(video_info, sort_keys=True),
    }
    save_file({"latents": latents}, str(output_path), metadata=metadata)
    return {
        "status": "ok",
        "video": str(video_path),
        "latents": str(output_path),
        "frames": int(video.shape[1]),
        "latent_shape": tuple(latents.shape),
    }


def add_wan_vae_latents_to_dataset(
    dataset_root: str | Path = DATASET_ROOT,
    checkpoint_dir: str | Path = WAN_CHECKPOINT_DIR,
    wan_repo_root: str | Path = WAN_REPO_ROOT,
    device: Optional[str | torch.device] = None,
    vae_dtype: torch.dtype = torch.float32,
    save_dtype: torch.dtype = torch.float16,
    overwrite: bool = False,
    limit: Optional[int] = None,
    show_progress: bool = True,
    continue_on_error: bool = True,
    expected_size: Tuple[int, int] = DEFAULT_WAN_EXPECTED_SIZE,
    resize: bool = False,
) -> list[Dict]:
    """Scan dataset_root/*/video.mp4 and add latents.safetensors to each subfolder."""
    dataset_root = Path(dataset_root)
    all_video_paths = sorted(dataset_root.glob("*/video.mp4"))
    if limit is not None:
        all_video_paths = all_video_paths[: int(limit)]
    if not all_video_paths:
        return []

    records = []
    video_paths = []
    for video_path in all_video_paths:
        latents_path = video_path.with_name("latents.safetensors")
        if latents_path.exists() and not overwrite:
            records.append({"status": "skipped", "reason": "exists", "video": str(video_path), "latents": str(latents_path)})
        else:
            video_paths.append(video_path)
    if not video_paths:
        return records

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu") if device is None else torch.device(device)
    vae = load_wan_ti2v_5b_vae(checkpoint_dir, wan_repo_root, device=device, dtype=vae_dtype)

    iterator = video_paths
    if show_progress:
        try:
            from tqdm.auto import tqdm
            iterator = tqdm(video_paths, desc="Encoding Wan VAE latents")
        except Exception:
            pass

    for video_path in iterator:
        try:
            record = encode_video_to_wan_latents(
                video_path,
                vae=vae,
                checkpoint_dir=checkpoint_dir,
                wan_repo_root=wan_repo_root,
                device=device,
                vae_dtype=vae_dtype,
                save_dtype=save_dtype,
                overwrite=overwrite,
                expected_size=expected_size,
                resize=resize,
            )
        except Exception as exc:
            if not continue_on_error:
                raise
            record = {"status": "error", "video": str(video_path), "error": repr(exc)}
        records.append(record)
    return records


# Example full pass:
records = add_wan_vae_latents_to_dataset()

# Example smoke test:
# records = add_wan_vae_latents_to_dataset(limit=1)


KeyboardInterrupt: 

In [1]:
# Debug: simulate and display 10 realtime Navier-Stokes videos.
import html
import importlib.util
import json
import warnings
from pathlib import Path

import torch
from IPython.display import HTML, Video, display
from torchvision.io import write_video

def _load_dataset_generator_module():
    candidates = [
        Path("/home/azureuser/physics/navier/generate_navier_dataset_with_latents_new.py"),
        Path.cwd() / "generate_navier_dataset_with_latents_new.py",
        Path.cwd() / "navier" / "generate_navier_dataset_with_latents_new.py",
    ]
    candidates.extend(
        parent / "generate_navier_dataset_with_latents_new.py"
        for parent in Path.cwd().resolve().parents
    )
    candidates.extend(
        parent / "navier" / "generate_navier_dataset_with_latents_new.py"
        for parent in Path.cwd().resolve().parents
    )
    for module_path in candidates:
        if module_path.exists():
            spec = importlib.util.spec_from_file_location("navier_dataset_generator", module_path)
            if spec is None or spec.loader is None:
                raise ImportError(f"Could not load module spec for {module_path}")
            module = importlib.util.module_from_spec(spec)
            spec.loader.exec_module(module)
            return module
    raise FileNotFoundError("Could not find generate_navier_dataset_with_latents_new.py")

_navier_generator = _load_dataset_generator_module()
make_initial_velocity_field = _navier_generator.make_initial_velocity_field
simulate_navier_stokes = _navier_generator.simulate_navier_stokes
simulate_navier_stokes_penalized_spectral_batch = _navier_generator.simulate_navier_stokes_penalized_spectral_batch
_realtime_video_sample_times = _navier_generator._realtime_video_sample_times
_simulation_to_uint8_video_frames = _navier_generator._simulation_to_uint8_video_frames
_verify_realtime_video_tensor = _navier_generator._verify_realtime_video_tensor
_verify_written_video_timing = _navier_generator._verify_written_video_timing
OBSTACLE_NORMALIZED_BOX = _navier_generator.OBSTACLE_NORMALIZED_BOX
OBSTACLE_LEFT_PADDING = _navier_generator.OBSTACLE_LEFT_PADDING
OBSTACLE_SIZE_SCALE = _navier_generator.OBSTACLE_SIZE_SCALE

DEBUG_VIDEO_COUNT = 10
DEBUG_BATCH_SIZE = 2
DEBUG_FPS = 64
DEBUG_DURATION_SECONDS = 2.5
DEBUG_GRID_SIZE = 256
DEBUG_VIDEO_WIDTH = 480
DEBUG_VIDEO_HEIGHT = 240
DEBUG_FIELD = "speed"
DEBUG_WIND_SPEED = 2
DEBUG_NU = 1e-3
DEBUG_RHO = 1.0
DEBUG_MAX_DELTA_T = 5e-3
DEBUG_OBSTACLE_METHOD = "penalized_spectral"
DEBUG_MASKED_PRESSURE_MAX_ITERATIONS = 50
DEBUG_MASKED_PRESSURE_TOLERANCE = 1e-3
DEBUG_INITIAL_MASKED_PRESSURE_MAX_ITERATIONS = 120
DEBUG_INITIAL_MASKED_PRESSURE_TOLERANCE = 1e-5
# Keep this at 0 for honest realtime: frame 1 is exactly 1 / DEBUG_FPS seconds after frame 0.
DEBUG_INITIAL_PROJECTION_STEPS = 0
DEBUG_USE_COMPILE = True #torch.cuda.is_available()
DEBUG_SEED = 20_260_623
DEBUG_DATASET_ROOT = Path(globals().get("DATASET_ROOT", "/home/azureuser/datasets/navier_fast"))
DEBUG_OUTPUT_DIR = DEBUG_DATASET_ROOT / f"debug_{DEBUG_DURATION_SECONDS:g}s_{DEBUG_FPS}fps_{DEBUG_VIDEO_WIDTH}x{DEBUG_VIDEO_HEIGHT}"
DEBUG_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DEBUG_DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEBUG_SAMPLE_TIMES = _realtime_video_sample_times(
    DEBUG_DURATION_SECONDS,
    DEBUG_FPS,
    torch.device("cpu"),
    torch.float64,
)
DEBUG_SIMULATION_SAMPLE_TIMES = DEBUG_SAMPLE_TIMES
DEBUG_EXPECTED_FRAMES = int(DEBUG_SAMPLE_TIMES.numel()) + 1
DEBUG_VIDEO_OPTIONS = {"crf": "18", "preset": "veryfast"}

debug_records = []
debug_batches = [
    list(range(start, min(DEBUG_VIDEO_COUNT, start + DEBUG_BATCH_SIZE)))
    for start in range(0, DEBUG_VIDEO_COUNT, DEBUG_BATCH_SIZE)
]
try:
    from tqdm.auto import tqdm

    debug_iterator = tqdm(debug_batches, desc="Simulating debug batches")
except Exception:
    debug_iterator = debug_batches

for debug_batch in debug_iterator:
    batch_items = []
    for debug_index in debug_batch:
        sample_seed = DEBUG_SEED + debug_index
        velocity0, _, obstacle = make_initial_velocity_field(
            n=DEBUG_GRID_SIZE,
            amplitude=DEBUG_WIND_SPEED,
            seed=sample_seed,
            device=DEBUG_DEVICE,
            return_obstacle=True,
        )
        batch_items.append(
            {
                "index": debug_index,
                "seed": sample_seed,
                "velocity0": velocity0,
                "obstacle": obstacle,
            }
        )

    if DEBUG_OBSTACLE_METHOD == "penalized_spectral" and len(batch_items) > 1:
        simulations = simulate_navier_stokes_penalized_spectral_batch(
            torch.stack([item["velocity0"] for item in batch_items], dim=0),
            nu=DEBUG_NU,
            rho=DEBUG_RHO,
            T=DEBUG_DURATION_SECONDS,
            max_delta_t=DEBUG_MAX_DELTA_T,
            obstacle_mask=torch.stack([item["obstacle"]["mask"] for item in batch_items], dim=0),
            obstacle_penalty_eta=1e-3,
            project_initial=True,
            boundary_u=DEBUG_WIND_SPEED,
            boundary_v=0.0,
            sample_times=DEBUG_SIMULATION_SAMPLE_TIMES,
        )
    else:
        simulations = [
            simulate_navier_stokes(
                item["velocity0"],
                nu=DEBUG_NU,
                rho=DEBUG_RHO,
                T=DEBUG_DURATION_SECONDS,
                max_delta_t=DEBUG_MAX_DELTA_T,
                pressure_method="spectral",
                use_compile=DEBUG_USE_COMPILE,
                obstacle_mask=item["obstacle"]["mask"],
                obstacle_method=DEBUG_OBSTACLE_METHOD,
                masked_pressure_max_iterations=DEBUG_MASKED_PRESSURE_MAX_ITERATIONS,
                masked_pressure_tolerance=DEBUG_MASKED_PRESSURE_TOLERANCE,
                initial_masked_pressure_max_iterations=DEBUG_INITIAL_MASKED_PRESSURE_MAX_ITERATIONS,
                initial_masked_pressure_tolerance=DEBUG_INITIAL_MASKED_PRESSURE_TOLERANCE,
                initial_projection_steps=DEBUG_INITIAL_PROJECTION_STEPS,
                boundary_u=DEBUG_WIND_SPEED,
                boundary_v=0.0,
                sample_times=DEBUG_SIMULATION_SAMPLE_TIMES,
            )
            for item in batch_items
        ]

    for item, simulation in zip(batch_items, simulations):
        debug_index = item["index"]
        sample_seed = item["seed"]
        velocity0 = item["velocity0"]
        obstacle = item["obstacle"]
        initial_frames, _ = _simulation_to_uint8_video_frames(
            {
                "velocity": velocity0.detach().unsqueeze(0),
                "obstacle_mask": obstacle["mask"],
            },
            field=DEBUG_FIELD,
            output_size=(DEBUG_VIDEO_HEIGHT, DEBUG_VIDEO_WIDTH),
            velocity_color_bound=DEBUG_WIND_SPEED,
        )
        simulation_frames, _ = _simulation_to_uint8_video_frames(
            simulation,
            field=DEBUG_FIELD,
            output_size=(DEBUG_VIDEO_HEIGHT, DEBUG_VIDEO_WIDTH),
            velocity_color_bound=DEBUG_WIND_SPEED,
        )
        frames = torch.cat((initial_frames, simulation_frames), dim=0)
        _verify_realtime_video_tensor(
            frames,
            DEBUG_DURATION_SECONDS,
            DEBUG_FPS,
            expected_frame_count=DEBUG_EXPECTED_FRAMES,
            label=f"debug video {debug_index}",
        )

        video_path = DEBUG_OUTPUT_DIR / f"debug_{debug_index:02d}.mp4"
        with warnings.catch_warnings():
            warnings.filterwarnings("ignore", category=UserWarning, module="torchvision")
            write_video(
                str(video_path),
                frames,
                fps=DEBUG_FPS,
                video_codec="libx264",
                options=DEBUG_VIDEO_OPTIONS,
            )
        timing = _verify_written_video_timing(
            video_path,
            DEBUG_DURATION_SECONDS,
            DEBUG_FPS,
            expected_frame_count=DEBUG_EXPECTED_FRAMES,
            label=f"debug video {debug_index}",
        )
        simulation_time_cpu = simulation["time"].detach().cpu()
        debug_time = torch.cat((simulation_time_cpu.new_zeros(1), simulation_time_cpu))
        debug_records.append(
            {
                "index": debug_index,
                "seed": sample_seed,
                "video": str(video_path),
                "frames": int(frames.shape[0]),
                "height": int(frames.shape[1]),
                "width": int(frames.shape[2]),
                "fps": DEBUG_FPS,
                "duration_seconds": DEBUG_DURATION_SECONDS,
                "wind_speed": float(DEBUG_WIND_SPEED),
                "batch_size": int(DEBUG_BATCH_SIZE),
                "first_frame": "raw_constant_wind_condition_frame",
                "second_frame": "projected_simulation_time_zero",
                "condition_frame_is_extra": True,
                "obstacle_normalized_box": OBSTACLE_NORMALIZED_BOX,
                "obstacle_left_padding": OBSTACLE_LEFT_PADDING,
                "obstacle_size_scale": OBSTACLE_SIZE_SCALE,
                "simulation_time": debug_time.tolist(),
                "timing": timing,
            }
        )

(DEBUG_OUTPUT_DIR / "debug_records.json").write_text(json.dumps(debug_records, indent=2))

video_cards = []
for record in debug_records:
    video_html = Video(
        filename=record["video"],
        embed=True,
        width=DEBUG_VIDEO_WIDTH,
        html_attributes="controls loop muted playsinline",
    )._repr_html_()
    caption = html.escape(
        f"#{record['index']:02d} seed={record['seed']} | "
        f"{record['frames']} frames @ {record['fps']} fps | {record['width']}x{record['height']}"
    )
    video_cards.append(
        f"<figure style='margin:0'>{video_html}"
        f"<figcaption style='font:12px sans-serif;margin-top:4px'>{caption}</figcaption>"
        "</figure>"
    )

display(
    HTML(
        "<div style='display:grid;grid-template-columns:repeat(2,minmax(320px,1fr));"
        "gap:16px;align-items:start'>"
        + "".join(video_cards)
        + "</div>"
    )
)

debug_records


/opt/miniforge/envs/new/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Simulating debug batches: 100%|██████████| 5/5 [00:20<00:00,  4.08s/it]


[{'index': 0,
  'seed': 20260623,
  'video': '/home/azureuser/datasets/navier/debug_2.5s_64fps_480x240/debug_00.mp4',
  'frames': 161,
  'height': 240,
  'width': 480,
  'fps': 64,
  'duration_seconds': 2.5,
  'wind_speed': 2.0,
  'batch_size': 2,
  'first_frame': 'raw_constant_wind_condition_frame',
  'second_frame': 'projected_simulation_time_zero',
  'condition_frame_is_extra': True,
  'obstacle_normalized_box': ((0.1, 0.33099999999999996),
   (0.3845, 0.6154999999999999)),
  'obstacle_left_padding': 0.1,
  'obstacle_size_scale': 0.7,
  'simulation_time': [0.0,
   0.0,
   0.015625,
   0.03125,
   0.046875,
   0.0625,
   0.078125,
   0.09375,
   0.109375,
   0.125,
   0.140625,
   0.15625,
   0.171875,
   0.1875,
   0.203125,
   0.21875,
   0.234375,
   0.25,
   0.265625,
   0.28125,
   0.296875,
   0.3125,
   0.328125,
   0.34375,
   0.359375,
   0.375,
   0.390625,
   0.40625,
   0.421875,
   0.4375,
   0.453125,
   0.46875,
   0.484375,
   0.5,
   0.515625,
   0.53125,
   0.546875